# Exploratory Data Analysis - Auction Data

**Objective:** Analyze auction-level data to understand patterns and propose engineered features for predicting item-level prices.

**Dataset:** `jpearce610/auction_data` from Hugging Face

**Goal:** The target task is to predict **item-level winning prices** (not in this dataset). This auction-level data will be used to create features that can be joined with item-level data.

---

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully")

In [ ]:
# Load the auction data
df = pd.read_parquet('../../../data/external/auction_data.parquet')

print(f"Dataset loaded: {len(df):,} rows, {len(df.columns)} columns")
print(f"\nColumns: {list(df.columns)}")

## 2. Data Overview

In [ ]:
# Display basic info
print("=" * 80)
print("DATA TYPES")
print("=" * 80)
print(df.dtypes)
print("\n" + "=" * 80)
print("BASIC STATISTICS")
print("=" * 80)
print(df.describe())

In [ ]:
# First few rows
print("First 5 rows of the dataset:")
df.head()

## 3. Data Quality Assessment

In [ ]:
# Missing values analysis
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percentage': missing_pct
}).sort_values('Missing_Count', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0])
print(f"\nColumns with no missing values: {len(missing_df[missing_df['Missing_Count'] == 0])}")

In [ ]:
# Visualize missing values
if missing_df['Missing_Count'].sum() > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    missing_cols = missing_df[missing_df['Missing_Count'] > 0]
    ax.barh(missing_cols.index, missing_cols['Missing_Percentage'])
    ax.set_xlabel('Missing Percentage (%)')
    ax.set_title('Missing Values by Column')
    plt.tight_layout()
    plt.show()
else:
    print("✓ No missing values in the dataset!")

## 4. Temporal Analysis

In [ ]:
# Parse datetime columns
df['auction_starts_dt'] = pd.to_datetime(df['auction_starts'])
df['auction_ends_dt'] = pd.to_datetime(df['auction_ends'])
df['auction_last_item_closes_dt'] = pd.to_datetime(df['auction_last_item_closes'])

# Calculate durations
df['auction_duration_hours'] = (df['auction_ends_dt'] - df['auction_starts_dt']).dt.total_seconds() / 3600
df['auction_extended_hours'] = (df['auction_last_item_closes_dt'] - df['auction_ends_dt']).dt.total_seconds() / 3600

# Extract temporal features
df['start_year'] = df['auction_starts_dt'].dt.year
df['start_month'] = df['auction_starts_dt'].dt.month
df['start_day_of_week'] = df['auction_starts_dt'].dt.dayofweek
df['start_hour'] = df['auction_starts_dt'].dt.hour

print("Temporal features created:")
print(f"  - Auction duration (hours)")
print(f"  - Extended bidding duration (hours)")
print(f"  - Year, Month, Day of Week, Hour of start")

In [ ]:
# Temporal distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Auctions by year
df['start_year'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Auctions by Year')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Count')

# Auctions by month
df['start_month'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 1], color='lightcoral')
axes[0, 1].set_title('Auctions by Month')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Count')

# Auctions by day of week
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_counts = df['start_day_of_week'].value_counts().sort_index()
axes[1, 0].bar(range(7), [day_counts.get(i, 0) for i in range(7)], color='lightgreen')
axes[1, 0].set_xticks(range(7))
axes[1, 0].set_xticklabels(day_names)
axes[1, 0].set_title('Auctions by Day of Week')
axes[1, 0].set_ylabel('Count')

# Auction duration distribution
axes[1, 1].hist(df['auction_duration_hours'], bins=50, color='plum', edgecolor='black')
axes[1, 1].set_title('Auction Duration Distribution')
axes[1, 1].set_xlabel('Duration (hours)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"\nAuction Duration Statistics:")
print(df['auction_duration_hours'].describe())

## 5. Auction Characteristics Analysis

In [ ]:
# Key auction metrics
print("=" * 80)
print("AUCTION METRICS SUMMARY")
print("=" * 80)

metrics = [
    'auction_item_count',
    'auction_total_viewed',
    'auction_total_winning_price',
    'auction_total_bid_count',
    'auction_total_images'
]

for metric in metrics:
    print(f"\n{metric}:")
    print(df[metric].describe())

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, metric in enumerate(metrics):
    axes[idx].hist(df[metric], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{metric.replace("auction_", "").replace("_", " ").title()}')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].axvline(df[metric].median(), color='red', linestyle='--', label='Median')
    axes[idx].legend()

# Remove extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

In [ ]:
# Extended bidding analysis
print("=" * 80)
print("EXTENDED BIDDING ANALYSIS")
print("=" * 80)

extended_count = df['auction_extended_bidding'].sum()
extended_pct = (extended_count / len(df)) * 100

print(f"Auctions with extended bidding: {extended_count:,} ({extended_pct:.1f}%)")
print(f"\nExtended bidding interval: {df['auction_extended_bidding_interval'].unique()}")
print(f"Extended bidding threshold: {df['auction_extended_bidding_threshold'].unique()}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
extended_counts = df['auction_extended_bidding'].value_counts()
ax.pie(extended_counts, labels=['Extended', 'Not Extended'], autopct='%1.1f%%', startangle=90)
ax.set_title('Percentage of Auctions with Extended Bidding')
plt.show()

## 6. Derived Metrics and Relationships

In [ ]:
# Create derived metrics
df['avg_price_per_item'] = df['auction_total_winning_price'] / df['auction_item_count'].replace(0, np.nan)
df['avg_views_per_item'] = df['auction_total_viewed'] / df['auction_item_count'].replace(0, np.nan)
df['avg_bids_per_item'] = df['auction_total_bid_count'] / df['auction_item_count'].replace(0, np.nan)
df['avg_images_per_item'] = df['auction_total_images'] / df['auction_item_count'].replace(0, np.nan)
df['bid_to_view_ratio'] = df['auction_total_bid_count'] / df['auction_total_viewed'].replace(0, np.nan)
df['win_rate'] = (df['auction_total_winning_price'] > 0).astype(int)  # Binary: did auction have any wins

print("Derived metrics created:")
print("  - Average price per item")
print("  - Average views per item")
print("  - Average bids per item")
print("  - Average images per item")
print("  - Bid to view ratio")
print("  - Win rate (binary)")

In [ ]:
# Correlation analysis
print("=" * 80)
print("CORRELATION ANALYSIS")
print("=" * 80)

numerical_cols = [
    'auction_item_count',
    'auction_total_viewed',
    'auction_total_winning_price',
    'auction_total_bid_count',
    'auction_total_images',
    'auction_duration_hours',
    'avg_price_per_item',
    'avg_views_per_item',
    'avg_bids_per_item',
    'avg_images_per_item',
    'bid_to_view_ratio'
]

corr_matrix = df[numerical_cols].corr()

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
ax.set_title('Correlation Matrix of Auction Metrics', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Show top correlations with total winning price
print("\nTop correlations with auction_total_winning_price:")
price_corr = corr_matrix['auction_total_winning_price'].sort_values(ascending=False)
print(price_corr)

In [ ]:
# Scatter plots of key relationships
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Item count vs total price
axes[0, 0].scatter(df['auction_item_count'], df['auction_total_winning_price'], alpha=0.3)
axes[0, 0].set_xlabel('Item Count')
axes[0, 0].set_ylabel('Total Winning Price')
axes[0, 0].set_title('Item Count vs Total Winning Price')

# Views vs bids
axes[0, 1].scatter(df['auction_total_viewed'], df['auction_total_bid_count'], alpha=0.3, color='orange')
axes[0, 1].set_xlabel('Total Views')
axes[0, 1].set_ylabel('Total Bid Count')
axes[0, 1].set_title('Views vs Bids')

# Avg images per item vs avg price per item
valid_data = df[['avg_images_per_item', 'avg_price_per_item']].dropna()
axes[1, 0].scatter(valid_data['avg_images_per_item'], valid_data['avg_price_per_item'], alpha=0.3, color='green')
axes[1, 0].set_xlabel('Avg Images per Item')
axes[1, 0].set_ylabel('Avg Price per Item')
axes[1, 0].set_title('Images vs Price (per item)')

# Duration vs total price
axes[1, 1].scatter(df['auction_duration_hours'], df['auction_total_winning_price'], alpha=0.3, color='purple')
axes[1, 1].set_xlabel('Auction Duration (hours)')
axes[1, 1].set_ylabel('Total Winning Price')
axes[1, 1].set_title('Duration vs Total Winning Price')

plt.tight_layout()
plt.show()

## 7. Outlier Detection

In [ ]:
# Box plots for key metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

key_metrics = [
    'auction_item_count',
    'auction_total_viewed',
    'auction_total_winning_price',
    'auction_total_bid_count',
    'avg_price_per_item',
    'auction_duration_hours'
]

for idx, metric in enumerate(key_metrics):
    axes[idx].boxplot(df[metric].dropna(), vert=True)
    axes[idx].set_title(f'{metric.replace("auction_", "").replace("_", " ").title()}')
    axes[idx].set_ylabel('Value')
    
    # Add statistics
    q1 = df[metric].quantile(0.25)
    q3 = df[metric].quantile(0.75)
    iqr = q3 - q1
    outliers = ((df[metric] < (q1 - 1.5 * iqr)) | (df[metric] > (q3 + 1.5 * iqr))).sum()
    axes[idx].text(0.5, 0.95, f'Outliers: {outliers}', 
                   transform=axes[idx].transAxes, 
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 8. Text Feature Analysis

In [ ]:
# Analyze text fields
print("=" * 80)
print("TEXT FEATURE ANALYSIS")
print("=" * 80)

# Title length
df['title_length'] = df['auction_title'].str.len()
df['title_word_count'] = df['auction_title'].str.split().str.len()

# Intro length
df['intro_length'] = df['auction_intro'].fillna('').str.len()
df['intro_word_count'] = df['auction_intro'].fillna('').str.split().str.len()

# Removal info length
df['removal_info_length'] = df['auction_removal_info'].fillna('').str.len()

print("\nTitle Statistics:")
print(df['title_length'].describe())
print(f"\nIntro Statistics:")
print(df['intro_length'].describe())
print(f"\nRemoval Info Statistics:")
print(df['removal_info_length'].describe())

In [ ]:
# Extract location from title (common pattern: "City (Province, Country)")
import re

def extract_location(title):
    """Extract city from auction title."""
    match = re.search(r'^([^(]+)\(', title)
    if match:
        return match.group(1).strip()
    return None

df['location'] = df['auction_title'].apply(extract_location)

print("\nTop 15 Auction Locations:")
location_counts = df['location'].value_counts().head(15)
print(location_counts)

# Visualize top locations
fig, ax = plt.subplots(figsize=(12, 6))
location_counts.plot(kind='barh', ax=ax, color='teal')
ax.set_xlabel('Number of Auctions')
ax.set_title('Top 15 Auction Locations')
plt.tight_layout()
plt.show()

## 9. Feature Engineering Recommendations

Based on the exploratory analysis, here are proposed engineered features for predicting item-level winning prices:

In [ ]:
# Summary of all engineered features
print("=" * 80)
print("PROPOSED ENGINEERED FEATURES FOR ITEM PRICE PREDICTION")
print("=" * 80)

features_summary = {
    "Auction-Level Aggregates": [
        "auction_total_winning_price - Total revenue for the auction",
        "auction_item_count - Number of items in auction",
        "auction_total_viewed - Total views across all items",
        "auction_total_bid_count - Total bids across all items",
        "auction_total_images - Total images across all items"
    ],
    "Auction-Level Averages (Contextual Features)": [
        "avg_price_per_item - Average winning price per item in auction",
        "avg_views_per_item - Average views per item in auction",
        "avg_bids_per_item - Average bids per item in auction",
        "avg_images_per_item - Average images per item in auction",
        "bid_to_view_ratio - Engagement rate (bids/views)"
    ],
    "Temporal Features": [
        "auction_duration_hours - Duration of auction in hours",
        "auction_extended_hours - Hours extended beyond scheduled end",
        "start_year, start_month - Seasonality indicators",
        "start_day_of_week - Day of week effect",
        "start_hour - Time of day effect"
    ],
    "Auction Mechanism Features": [
        "auction_extended_bidding - Boolean: soft-close enabled",
        "auction_extended_bidding_threshold - Minutes before close for extension",
        "auction_extended_bidding_interval - Extension duration in minutes"
    ],
    "Location Features": [
        "location - Extracted city from title",
        "location_encoded - One-hot or target encoding of location"
    ],
    "Text Features": [
        "title_length - Character length of title",
        "title_word_count - Word count in title",
        "intro_length - Character length of description",
        "removal_info_length - Length of pickup instructions",
        "has_partner_url - Boolean: affiliated auction"
    ],
    "Relative Position Features (for item-level prediction)": [
        "item_position_in_auction - Position/rank within auction",
        "item_pct_through_auction - Temporal position (0-1)",
        "item_price_vs_auction_avg - Item price relative to auction average",
        "item_views_vs_auction_avg - Item views relative to auction average"
    ]
}

for category, features in features_summary.items():
    print(f"\n{category}:")
    for feature in features:
        print(f"  • {feature}")

## 10. Key Insights and Recommendations

In [ ]:
print("=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

insights = f"""
1. DATA QUALITY
   ✓ Dataset contains {len(df):,} auctions with complete data
   ✓ No missing values in key metrics
   ✓ Date range: {df['start_year'].min()} to {df['start_year'].max()}

2. AUCTION CHARACTERISTICS
   • Median items per auction: {df['auction_item_count'].median():.0f}
   • Median total winning price: ${df['auction_total_winning_price'].median():,.2f}
   • {extended_pct:.1f}% of auctions use extended bidding (soft-close)
   • Typical auction duration: {df['auction_duration_hours'].median():.1f} hours

3. ENGAGEMENT METRICS
   • Strong correlation between views and bids (r={corr_matrix.loc['auction_total_viewed', 'auction_total_bid_count']:.2f})
   • Item count highly correlated with total price (r={corr_matrix.loc['auction_item_count', 'auction_total_winning_price']:.2f})
   • Average bid-to-view ratio: {df['bid_to_view_ratio'].median():.3f}

4. GEOGRAPHIC DISTRIBUTION
   • Top location: {location_counts.index[0]} ({location_counts.iloc[0]} auctions)
   • {len(df['location'].unique())} unique locations

5. TEMPORAL PATTERNS
   • Most auctions start in specific time windows
   • Day of week shows variation in auction frequency
   • Extended bidding adds avg {df['auction_extended_hours'].mean():.2f} hours
"""

print(insights)

## 11. Save Processed Data with Engineered Features

In [ ]:
# Select final feature set
feature_columns = [
    # Original columns
    'auction_id',
    'auction_title',
    'auction_starts',
    'auction_ends',
    'auction_item_count',
    'auction_total_viewed',
    'auction_total_winning_price',
    'auction_total_bid_count',
    'auction_total_images',
    'auction_extended_bidding',
    'auction_extended_bidding_threshold',
    # Engineered features
    'auction_duration_hours',
    'auction_extended_hours',
    'start_year',
    'start_month',
    'start_day_of_week',
    'start_hour',
    'avg_price_per_item',
    'avg_views_per_item',
    'avg_bids_per_item',
    'avg_images_per_item',
    'bid_to_view_ratio',
    'location',
    'title_length',
    'title_word_count',
    'intro_length'
]

df_processed = df[feature_columns].copy()

# Save to parquet
output_path = '../../../data/processed/auction_data_with_features.parquet'
df_processed.to_parquet(output_path, index=False)

print(f"✓ Processed data saved to: {output_path}")
print(f"  Rows: {len(df_processed):,}")
print(f"  Columns: {len(df_processed.columns)}")

---

## Summary

This exploratory analysis has provided insights into auction-level characteristics and proposed engineered features for item-level price prediction. The key findings include:

1. **High-quality data** with no missing values in critical fields
2. **Strong correlations** between engagement metrics (views, bids) and prices
3. **Temporal patterns** that can be leveraged as features
4. **Geographic variation** across auction locations
5. **Auction context** matters - average metrics can serve as powerful features

The engineered features are designed to capture:
- Auction-level context (avg prices, engagement)
- Temporal effects (seasonality, time of day)
- Auction mechanics (extended bidding)
- Geographic factors
- Relative positioning of items within auctions

**Next Steps:**
1. Join these auction-level features with item-level data
2. Create item-relative features (e.g., price vs auction average)
3. Build and evaluate predictive models
4. Consider interaction features between auction and item characteristics
---